## Objetivo de este Smoke Test
- M1: que el batch traiga imágenes (3,448,448), máscaras (1,448,448) binarias, y que haya 40 glaucomas en train.
- M2 (a–d): que los 3 backbones hagan forward sin errores, que predict dé una distribución que suma 1, que get_gradcam salga (448,448) dentro de [0,1], y que save/load reproduzcan exactamente la predicción.

In [ ]:
# ============================================================
# SMOKE TEST: M1 (DataModule) + M2 (CNNClassifier)
# ============================================================
import sys, logging
from pathlib import Path

# Agrega el dir del experimento al path. Ajusta si NO corres desde notebook/.
sys.path.insert(0, str(Path("..").resolve()))

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
import yaml, torch, numpy as np
from modules.data_module import DataModule, _EXPERIMENT_DIR   # _EXPERIMENT_DIR = fuente de verdad
from modules.cnn_classifier import CNNClassifier

# --- Config (ruta robusta, independiente del working dir) ---
config = yaml.safe_load((_EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8"))

# --- Asegurar annotations.json / splits.json (si faltan, convertir REFUGE) ---
if not (_EXPERIMENT_DIR / config["data"]["output_dir"] / "annotations.json").exists():
    print("annotations.json no existe → ejecutando convert_refuge_format...")
    from scripts.convert_refuge_format import main as convert_main
    convert_main()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ===================== M1: DataModule =====================
# Pasamos seed y augmentations para que vengan del config (no de los defaults).
data_cfg = {**config["data"], "seed": config["seed"], "augmentations": config["augmentations"]}
dm = DataModule(data_cfg)
print("Distribución de clases:", dm.get_class_distribution())
print("Glaucomas en train:", len(dm.get_glaucoma_indices("train")))

batch = next(iter(dm.get_val_loader()))
print("image:", tuple(batch["image"].shape), batch["image"].dtype,
      "| mask:", tuple(batch["mask"].shape), "| labels:", batch["label"][:8].tolist())

assert batch["image"].shape[1:] == (3, 448, 448), "imagen mal formada"
assert batch["mask"].shape[1:] == (1, 448, 448), "máscara mal formada"
assert set(np.unique(batch["mask"].numpy()).tolist()) <= {0.0, 1.0}, "máscara no binaria"
print("✅ M1 OK\n")

# ===================== M2: CNNClassifier (3 backbones) =====================
one_image = batch["image"][0]   # (3,448,448)

for backbone in config["backbones"]:
    print(f"── {backbone}")
    clf = CNNClassifier({"backbone": backbone,
                         "num_classes": config["classifier"]["num_classes"],
                         "pretrained": True, "seed": config["seed"]})

    # (a) forward de un batch -> logits (B, 2)
    clf.model.eval()
    with torch.no_grad():
        logits = clf.model(batch["image"].to(device))
    assert tuple(logits.shape) == (batch["image"].shape[0], 2), "logits mal formados"

    # (b) predict -> la distribución debe sumar 1
    pred = clf.predict(one_image)
    suma = sum(pred["distribution"].values())
    print(f"   predict: {pred['prediction']} {pred['distribution']} | suma={suma:.4f}")
    assert abs(suma - 1.0) < 1e-4, "la distribución no suma 1"

    # (c) get_gradcam -> heatmap (448,448) en [0,1]
    cam = clf.get_gradcam(one_image)
    print(f"   gradcam: shape={cam.shape} rango=[{cam.min():.3f}, {cam.max():.3f}]")
    assert cam.shape == (448, 448), "gradcam con forma incorrecta"
    assert cam.min() >= 0.0 and cam.max() <= 1.0 + 1e-6, "gradcam fuera de [0,1]"

    # (d) save / load -> predicciones idénticas
    ckpt = _EXPERIMENT_DIR / f"results/_smoke/{backbone}.pth"
    clf.save(ckpt)
    clf2 = CNNClassifier({"backbone": backbone,
                          "num_classes": config["classifier"]["num_classes"],
                          "pretrained": False, "seed": config["seed"]})
    clf2.load(ckpt)
    a, b = clf.predict(one_image)["distribution"], clf2.predict(one_image)["distribution"]
    assert all(abs(a[k] - b[k]) < 1e-5 for k in a), "save/load no coinciden"
    print("   ✅ forward / predict / gradcam / save-load OK")

print("\n🎉 Smoke test completo: M1 + M2 funcionan de punta a punta.")